# Evo-1 + VLABench (Colab)

Benchmarks **Evo-1** on **VLABench** entirely inside Google Colab.

This notebook will:
- Clone Evo-1 and VLABench
- Install dependencies in the Colab runtime (no conda)
- Download VLABench assets
- Download the Evo-1 LIBERO checkpoint (used here as a zero-shot policy)
- Start an Evo-1 WebSocket inference server
- Run a VLABench evaluation client for a small task list

Colab setup:
- **Runtime > Change runtime type > GPU**
- Recommended: L4/A100 for smoother runs

In [2]:
# 1. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Set Runtime > Change runtime type > GPU')
print('GPU:', torch.cuda.get_device_name(0))
mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'VRAM: {mem_gb:.1f} GB')

CUDA available: False


RuntimeError: No GPU detected. Set Runtime > Change runtime type > GPU

In [ ]:
# 2. System deps (MuJoCo / EGL / build tooling)
!apt-get update -qq
!apt-get install -y -qq git wget unzip ffmpeg \
  libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa libgles2-mesa \
  build-essential
print('✅ System dependencies installed')

In [ ]:
# 2.5 Headless rendering setup (no display; save files instead)
import os
import warnings

# Force headless/offscreen rendering (Colab has no GUI).
# MuJoCo can render offscreen via EGL; no X server needed.
os.environ.setdefault('MUJOCO_GL', 'egl')

# Some stacks (glfw) warn if DISPLAY is missing even when using offscreen rendering.
# This warning is harmless in headless Colab, so we silence it to keep logs clean.
warnings.filterwarnings('ignore', message=r'.*X11: The DISPLAY environment variable is missing.*')

# Keep DISPLAY unset/empty to avoid accidentally trying to open a window.
os.environ.pop('DISPLAY', None)

print('MUJOCO_GL =', os.environ.get('MUJOCO_GL'))
print('DISPLAY =', os.environ.get('DISPLAY'))

In [ ]:
# 3. Base Python deps (then install repos)
print('📦 Installing base Python packages...')
!pip install -q --upgrade pip
!pip install -q 'numpy>=1.26.4,<2.0' pillow opencv-python packaging tqdm scipy colorlog colorama
!pip install -q websockets websocket-client mediapy
!pip install -q huggingface_hub
# Evo-1 expects torch 2.5.1 + torchvision 0.20.1
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
print('✅ Base packages installed')

In [ ]:
# 4. Clone repos + install (Evo-1, VLABench, rrt-algorithms)
from pathlib import Path
import os
os.environ.pop('PYTHONPATH', None)

# Evo-1
if not Path('/content/Evo-1/.git').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
!pip install -q -r /content/Evo-1/Evo_1/requirements.txt

# VLABench dependency: rrt-algorithms
if not Path('/content/rrt-algorithms/.git').exists():
    !git clone https://github.com/motion-planning/rrt-algorithms.git /content/rrt-algorithms
!cd /content/rrt-algorithms && pip install -q -e .

# VLABench
if not Path('/content/VLABench/.git').exists():
    !git clone https://github.com/OpenMOSS/VLABench.git /content/VLABench

# Install VLABench deps and package (editable)
!cd /content/VLABench && pip install -q -r requirements.txt
!cd /content/VLABench && pip install -q -e .

print('✅ Repos cloned and installed')

In [ ]:
# 5. Download VLABench assets (official helper script)
from pathlib import Path
assets_dir = Path('/content/VLABench/VLABench/assets')
assets_dir.mkdir(parents=True, exist_ok=True)

# Heuristic: if obj/ and scenes/ already exist, skip
if (assets_dir / 'obj').exists() and (assets_dir / 'scenes').exists():
    print('✅ VLABench assets already present; skipping download')
else:
    print('📥 Downloading VLABench assets...')
    !cd /content/VLABench && python scripts/download_assets.py
    print('✅ VLABench assets download finished')

In [ ]:
# 6. Download Evo-1 checkpoint (LIBERO checkpoint used here as zero-shot)
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil

ckpt_dir = Path('/content/Evo-1/checkpoints/libero')
ckpt_dir.mkdir(parents=True, exist_ok=True)
model_file = ckpt_dir / 'mp_rank_00_model_states.pt'

if model_file.exists() and model_file.stat().st_size > 1_000_000:
    print(f'✅ Checkpoint already present ({model_file.stat().st_size/1e9:.2f} GB)')
else:
    if ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    print('📥 Downloading MINT-SJTU/Evo1_LIBERO ...')
    snapshot_download(
        repo_id='MINT-SJTU/Evo1_LIBERO',
        local_dir=str(ckpt_dir),
        local_dir_use_symlinks=False,
    )
    print('✅ Checkpoint download complete')

required = ['config.json', 'norm_stats.json', 'mp_rank_00_model_states.pt']
missing = [f for f in required if not (ckpt_dir / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files in checkpoint dir: {missing}')
print('✅ Checkpoint looks complete')

In [ ]:
# 7. Create Evo-1 WebSocket server (persistent connections + large message support)
from pathlib import Path

server_path = Path('/content/Evo-1/Evo_1/scripts/Evo1_multi_server.py')
server_path.parent.mkdir(parents=True, exist_ok=True)

server_code = r'''#!/usr/bin/env python3
import sys
import os
import asyncio
import json
import numpy as np
import torch
from PIL import Image
import websockets
import argparse

MAX_SIZE = 100 * 1024 * 1024  # 100MB

parser = argparse.ArgumentParser()
parser.add_argument('--port', type=int, required=True)
parser.add_argument('--checkpoint', type=str, required=True)
parser.add_argument('--name', type=str, default='evo1')
args = parser.parse_args()

print(f'[{args.name}] Starting on port {args.port}')
print(f'[{args.name}] Loading checkpoint from {args.checkpoint}')

# Ensure we can import scripts.Evo1 by adding Evo_1/ to sys.path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from scripts.Evo1 import EVO1

class Normalizer:
    def __init__(self, stats_or_path):
        if isinstance(stats_or_path, str):
            with open(stats_or_path, 'r') as f:
                stats = json.load(f)
        else:
            stats = stats_or_path

        def pad_to_24(x):
            x = torch.tensor(x, dtype=torch.float32)
            if x.shape[0] < 24:
                pad = torch.zeros(24 - x.shape[0], dtype=torch.float32)
                x = torch.cat([x, pad], dim=0)
            elif x.shape[0] > 24:
                raise ValueError(f'Input length {x.shape[0]} exceeds expected 24')
            return x

        if len(stats) != 1:
            raise ValueError(f'norm_stats.json should contain only one robot key, but: {list(stats.keys())}')
        robot_key = list(stats.keys())[0]
        robot_stats = stats[robot_key]

        self.state_min = pad_to_24(robot_stats['observation.state']['min'])
        self.state_max = pad_to_24(robot_stats['observation.state']['max'])
        self.action_min = pad_to_24(robot_stats['action']['min'])
        self.action_max = pad_to_24(robot_stats['action']['max'])

    def normalize_state(self, state: torch.Tensor) -> torch.Tensor:
        state_min = self.state_min.to(state.device, dtype=state.dtype)
        state_max = self.state_max.to(state.device, dtype=state.dtype)
        return torch.clamp(2 * (state - state_min) / (state_max - state_min + 1e-8) - 1, -1.0, 1.0)

    def denormalize_action(self, action: torch.Tensor) -> torch.Tensor:
        action_min = self.action_min.to(action.device, dtype=action.dtype)
        action_max = self.action_max.to(action.device, dtype=action.dtype)
        if action.ndim == 1:
            action = action.view(1, -1)
        return (action + 1.0) / 2.0 * (action_max - action_min + 1e-8) + action_min

def load_model_and_normalizer(ckpt_dir: str):
    config = json.load(open(os.path.join(ckpt_dir, 'config.json')))
    stats = json.load(open(os.path.join(ckpt_dir, 'norm_stats.json')))

    config['finetune_vlm'] = False
    config['finetune_action_head'] = False
    config['num_inference_timesteps'] = 32

    model = EVO1(config).eval()
    ckpt_path = os.path.join(ckpt_dir, 'mp_rank_00_model_states.pt')
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['module'], strict=True)

    device = config.get('device', 'cuda')
    model = model.to(device)
    normalizer = Normalizer(stats)
    return model, normalizer

print('Loading EVO1 model...')
model, normalizer = load_model_and_normalizer(args.checkpoint)
print(f'[{args.name}] EVO1 model loaded')

async def handle_client(websocket):
    client_id = id(websocket)
    print(f'[{args.name}] Client {client_id} connected')
    try:
        async for message in websocket:
            try:
                data = json.loads(message)

                images = []
                for img_data in data.get('image', []):
                    arr = np.array(img_data, dtype=np.uint8)
                    if arr.ndim == 3:
                        images.append(Image.fromarray(arr))
                while len(images) < 3:
                    images.append(Image.new('RGB', (448, 448), (0, 0, 0)))

                state = np.array(data.get('state', [0.0] * 7), dtype=np.float32)
                prompt = data.get('prompt', '')
                image_mask = data.get('image_mask', [1, 1, 0])
                action_mask = data.get('action_mask', [1] * 7 + [0] * 17)

                device = 'cuda' if torch.cuda.is_available() else 'cpu'
                image_mask_tensor = torch.tensor(image_mask, dtype=torch.int32, device=device)
                action_mask_tensor = torch.tensor([action_mask], dtype=torch.int32, device=device)

                state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
                if state_tensor.shape[0] < 24:
                    state_tensor = torch.cat([state_tensor, torch.zeros(24 - state_tensor.shape[0], device=device)])
                state_normalized = normalizer.normalize_state(state_tensor.unsqueeze(0))

                with torch.no_grad():
                    fused_tokens = model.get_vl_embeddings(
                        images=images,
                        image_mask=image_mask_tensor,
                        prompt=prompt
                    )
                    actions = model.predict_action(
                        fused_tokens=fused_tokens,
                        state=state_normalized,
                        action_mask=action_mask_tensor
                    )
                    if actions.ndim == 2 and actions.shape[1] == 1200:
                        actions = actions.view(1, 50, 24)

                    horizon = actions.shape[1]
                    denorm = []
                    for t in range(horizon):
                        a_t = actions[0, t, :]
                        a_t = normalizer.denormalize_action(a_t).squeeze(0)
                        denorm.append(a_t)
                    actions = torch.stack(denorm, dim=0)

                await websocket.send(json.dumps(actions.detach().cpu().numpy().tolist()))
            except Exception as e:
                await websocket.send(json.dumps({'error': str(e)}))
    except websockets.exceptions.ConnectionClosed:
        pass
    finally:
        print(f'[{args.name}] Client {client_id} disconnected')

async def main():
    print(f'[{args.name}] Server running at ws://0.0.0.0:{args.port}')
    async with websockets.serve(
        handle_client,
        '0.0.0.0',
        args.port,
        max_size=MAX_SIZE,
        ping_interval=30,
        ping_timeout=60,
        close_timeout=10,
    ):
        await asyncio.Future()

if __name__ == '__main__':
    asyncio.run(main())
'''

server_path.write_text(server_code)
!chmod +x /content/Evo-1/Evo_1/scripts/Evo1_multi_server.py
print('✅ Wrote /content/Evo-1/Evo_1/scripts/Evo1_multi_server.py')

In [ ]:
# 8. Create VLABench evaluation client
from pathlib import Path

client_path = Path('/content/vlabench_client.py')

client_code = r'''#!/usr/bin/env python3
import asyncio
import argparse
import json
import math
import os
import sys
import time
import numpy as np
import websockets

os.environ['MUJOCO_GL'] = 'egl'
sys.path.insert(0, '/content/VLABench')

from VLABench.robots import *  # noqa: F401,F403
from VLABench.tasks import *   # noqa: F401,F403
from VLABench.envs import load_env
from VLABench.utils.register import register

MAX_SIZE = 100 * 1024 * 1024
RECV_TIMEOUT_S = 30

def encode_image(img):
    return img.astype(np.uint8).tolist()

def quat2axisangle(q):
    q = np.array(q, dtype=np.float64)
    q[3] = max(-1, min(1, q[3]))
    den = np.sqrt(1 - q[3]**2)
    return np.zeros(3) if abs(den) < 1e-8 else (q[:3] * 2 * math.acos(q[3])) / den

def prepare_observation(obs, instruction):
    import cv2
    img = obs.get('rgb', np.zeros((448, 448, 3), dtype=np.uint8))
    if img.ndim == 4:
        img = img[2] if img.shape[0] > 2 else img[0]
    wrist = obs.get('wrist', np.zeros((448, 448, 3), dtype=np.uint8))
    if wrist.ndim == 4:
        wrist = wrist[-1]
    if img.shape[:2] != (448, 448):
        img = cv2.resize(img, (448, 448))
    if wrist.shape[:2] != (448, 448):
        wrist = cv2.resize(wrist, (448, 448))

    ee = obs.get('ee_state', np.zeros(8))
    if len(ee) >= 7:
        state = np.concatenate([ee[:3], quat2axisangle(ee[3:7]), [ee[7]] if len(ee) > 7 else [0]]).tolist()
    else:
        state = [0.0] * 7

    return {
        'image': [
            encode_image(img),
            encode_image(wrist),
            encode_image(np.zeros((448, 448, 3), dtype=np.uint8)),
        ],
        'state': state,
        'prompt': instruction,
        'image_mask': [1, 1, 0],
        'action_mask': [1] * 7 + [0] * 17,
    }

async def run_episode(ws, env, instruction, max_steps):
    t0 = time.time()
    print('  reset()... ', end='', flush=True)
    timestep = env.reset()
    print(f'done ({time.time() - t0:.1f}s)', flush=True)
    obs = timestep.observation
    done = False
    step = 0
    action_spec = env.action_spec()
    env_action_dim = action_spec.shape[0]

    while not done and step < max_steps:
        obs_dict = {
            'rgb': obs.get('rgb', np.zeros((4, 480, 480, 3))),
            'ee_state': obs.get('ee_state', np.zeros(8)),
            'wrist': obs.get('wrist', obs.get('rgb', np.zeros((4, 480, 480, 3)))),
        }
        await ws.send(json.dumps(prepare_observation(obs_dict, instruction)))
        try:
            result = await asyncio.wait_for(ws.recv(), timeout=RECV_TIMEOUT_S)
        except asyncio.TimeoutError:
            raise TimeoutError(f'No response from server within {RECV_TIMEOUT_S}s')
        action_list = json.loads(result)
        if isinstance(action_list, dict) and 'error' in action_list:
            raise RuntimeError(f"Server error: {action_list['error']}")
        actions = np.array(action_list)
        action = actions[0][:env_action_dim]
        t_step = time.time()
        timestep = env.step(action)
        if step == 0 or (step + 1) % 10 == 0:
            print(f'  step {step + 1}: env.step() {time.time() - t_step:.3f}s', flush=True)
        obs = timestep.observation
        done = timestep.last()
        step += 1

    success = False
    if hasattr(env, 'task'):
        if hasattr(env.task, 'check_success'):
            success = env.task.check_success(env.physics)
        elif hasattr(env.task, '_check_success'):
            success = env.task._check_success(env.physics)
        elif hasattr(env.task, 'is_success'):
            success = env.task.is_success()
    if not success and hasattr(env, 'is_success'):
        success = env.is_success()
    if not success and hasattr(timestep, 'reward') and timestep.reward is not None:
        success = float(timestep.reward) > 0.5
    return bool(success), step

async def evaluate_task(server_url, task_name, n_episodes, max_steps):
    results = {'task': task_name, 'episodes': [], 'success_rate': 0.0}
    successes = 0
    for _ in range(n_episodes):
        t0 = time.time()
        print(f'  load_env({task_name})... ', end='', flush=True)
        env = load_env(task_name, random_init=True, run_mode='eval', eval=False)
        print(f'done ({time.time() - t0:.1f}s)', flush=True)
        instruction = env.task.get_instruction()
        async with websockets.connect(server_url, max_size=MAX_SIZE, open_timeout=30) as ws:
            ok, steps = await run_episode(ws, env, instruction, max_steps)
        successes += int(ok)
        results['episodes'].append({'success': ok, 'steps': steps})
    results['success_rate'] = successes / n_episodes if n_episodes else 0.0
    return results

async def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--host', default='localhost')
    parser.add_argument('--port', type=int, default=9003)
    parser.add_argument('--tasks', nargs='+', default=['select_toy', 'select_fruit', 'select_drink'])
    parser.add_argument('--n-episodes', type=int, default=3)
    parser.add_argument('--max-steps', type=int, default=200)
    parser.add_argument('--save-dir', default='/content/logs/vlabench')
    args = parser.parse_args()

    server_url = f'ws://{args.host}:{args.port}'
    os.makedirs(args.save_dir, exist_ok=True)

    print('VLABench Evaluation for Evo-1')
    print('=' * 60)
    print('Server:', server_url)
    print('Tasks:', args.tasks)
    print('Episodes per task:', args.n_episodes)
    print(f'✅ Loaded {len(register._tasks)} VLABench tasks')
    print('=' * 60)

    all_results = {}
    for task in args.tasks:
        print(f'\nEvaluating: {task}')
        try:
            r = await evaluate_task(server_url, task, args.n_episodes, args.max_steps)
            all_results[task] = r
            print(f"  Success Rate: {r['success_rate'] * 100:.1f}%")
        except Exception as e:
            print('  ❌ Task failed:', e)
            all_results[task] = {'error': str(e), 'success_rate': 0.0}

    with open(os.path.join(args.save_dir, 'results.json'), 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f'\n✅ Saved results to {args.save_dir}/results.json')

if __name__ == '__main__':
    asyncio.run(main())
'''

client_path.write_text(client_code)
!chmod +x /content/vlabench_client.py
print('✅ Wrote /content/vlabench_client.py')

In [ ]:
# 9. Run Evo-1 server + VLABench client
import os
import subprocess
import time
from pathlib import Path

os.environ['MUJOCO_GL'] = 'egl'
os.environ['DISPLAY'] = ''

Path('/content/logs/vlabench').mkdir(parents=True, exist_ok=True)
PORT = 9003
CKPT = '/content/Evo-1/checkpoints/libero'

# Kill anything using the port / old server
subprocess.run(f"lsof -ti:{PORT} | xargs -r kill -9 2>/dev/null", shell=True)
subprocess.run("pkill -f 'Evo1_multi_server.py' 2>/dev/null", shell=True)
time.sleep(1)

server_cmd = f"cd /content/Evo-1/Evo_1 && python -u scripts/Evo1_multi_server.py --port {PORT} --checkpoint {CKPT} --name vlabench"
server_log_path = '/content/logs/vlabench/server.log'
client_log_path = '/content/logs/vlabench/client.log'

print('🚀 Starting Evo-1 server...')
server_log = open(server_log_path, 'w')
server_proc = subprocess.Popen(
    server_cmd,
    shell=True,
    stdout=server_log,
    stderr=server_log,
    env={**os.environ, 'PYTHONPATH': '/content/Evo-1/Evo_1', 'PYTHONUNBUFFERED': '1'}
)

# Wait for server
import websocket
def wait_ws(port, timeout_s=180):
    start = time.time()
    while time.time() - start < timeout_s:
        try:
            # NOTE: this is a websocket health-check and will appear in the server log
            # as a quick client connect/disconnect. This is expected.
            ws = websocket.create_connection(f"ws://localhost:{port}", timeout=5)
            ws.close()
            return True
        except Exception:
            time.sleep(3)
    return False

if not wait_ws(PORT):
    print('❌ Server did not come up in time. Last 40 log lines:')
    !tail -40 /content/logs/vlabench/server.log
    raise RuntimeError('Server failed to start')

print('✅ Server is ready')

print('🔬 Running VLABench client...')
# Quick sanity-check settings (faster feedback; increase later)
client_cmd = 'python -u /content/vlabench_client.py --host localhost --port 9003 --n-episodes 1 --max-steps 50'
client_log = open(client_log_path, 'w')
client_proc = subprocess.Popen(
    client_cmd,
    shell=True,
    stdout=client_log,
    stderr=client_log,
    env={**os.environ, 'PYTHONPATH': '/content/VLABench', 'PYTHONUNBUFFERED': '1'}
)

rc = client_proc.wait()
print('Client return code:', rc)

# Stop server
server_proc.terminate()
time.sleep(2)
try:
    server_proc.kill()
except Exception:
    pass
server_log.close()
client_log.close()

print('✅ Done')
print('Logs: /content/logs/vlabench/')

print('--- Server log (tail) ---')
!tail -40 /content/logs/vlabench/server.log 2>/dev/null || true

print('--- Client log (tail) ---')
!tail -80 /content/logs/vlabench/client.log 2>/dev/null || true

print('--- Client log (head) ---')
!head -80 /content/logs/vlabench/client.log 2>/dev/null || true

print('--- results.json ---')
!ls -la /content/logs/vlabench || true
!python -c "import json; print(json.dumps(json.load(open('/content/logs/vlabench/results.json')), indent=2))" 2>/dev/null || echo 'results.json not found'